# OpenAI 매개변수

# 개요
OpenAI 모델에 요청을 보낼 때 여러 매개변수를 사용하여 모델의 동작과 출력을 제어할 수 있습니다. \
이러한 매개변수를 이해하면 텍스트 생성, 질문 응답 또는 기타 사용 사례에 맞게 응답을 세부 조정할 수 있습니다.

더 자세한 예제는 공식 문서를 참조하세요: [Azure OpenAI Service](https://learn.microsoft.com/en-us/azure/ai-services/openai/reference)


In [1]:
import re
import requests
import sys
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import tiktoken
from dotenv import load_dotenv
load_dotenv()

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  azure_ad_token_provider=token_provider,
  api_version="2024-02-15-preview"
)

CHAT_COMPLETIONS_MODEL = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')
SEED = 123

# 매개변수: max_tokens
**설명**: 생성할 응답의 최대 토큰 수를 설정합니다. \
**기본값**: 16 \
**예제**: max_tokens=50

프롬프트의 토큰 수와 max_tokens의 합은 모델의 컨텍스트 길이를 초과할 수 없습니다. \
gpt-4o-mini의 경우 16,384 tokens이며 모델에 따라 다릅니다. 

In [3]:
def call_openai_with_max_tokens(max_tokens):
    response = client.chat.completions.create(
          model=CHAT_COMPLETIONS_MODEL,
          messages = [{"role":"system", "content":"You are a helpful assistant."},
                    {"role":"user","content": "최고의 반려동물은 "}],
                     max_tokens=max_tokens
                    
    )
    return response.choices[0].message.content

tokens = [32, 64, 120, 200]
for token in tokens:
    print(f"Max Tokens: {token}\n")
    print(call_openai_with_max_tokens(token))
    print("\n" + "-"*80 + "\n")

Max Tokens: 32

최고의 반려동물은 개인적인 취향, 라이프스타일, 주거 환경 등에 따라 매우 달라질 수 있습니다. 아래에

--------------------------------------------------------------------------------

Max Tokens: 64

"최고의 반려동물"이라는 질문에는 한 가지 정답이 없습니다. 반려동물을 고를 때는 개인의 라이프스타일, 주거 환경, 시간, 알레르기 여부, 원하는 상호작용 등에 따라 달라지기 때문입니다. 아래에 각 인기

--------------------------------------------------------------------------------

Max Tokens: 120

"최고의 반려동물"은 사람마다 다르게 생각할 수 있습니다. 각 가정의 환경, 생활 패턴, 알레르기 여부, 선호하는 활동, 시간, 예산 등에 따라 적합한 반려동물이 달라집니다. 아래에 대표적인 반려동물과 특징을 소개해드릴게요.

1. 강아지 (개)
- 특징: 충성심이 강하고 사람과 잘 어울림, 산책과 놀이가 필요함
- 장점: 활발하고 보호자와 유

--------------------------------------------------------------------------------

Max Tokens: 200

"최고의 반려동물"은 사람마다 다르며, 각자의 라이프스타일과 선호도에 따라 달라집니다. 여러 반려동물의 특징을 간단히 소개해드릴게요.

1. 강아지  
- 장점: 애정 표현이 많고, 충성심이 강하며, 함께 활동하기 좋음  
- 고려사항: 산책 등 꾸준한 운동과 시간, 훈련 필요

2. 고양이  
- 장점: 독립적이며, 집에서 키우기 쉬움, 관찰하는 재미  
- 고려사항: 은근한 관찰과 관리 필요, 털 관리

3. 햄스터/기니피그 등 작은 동물  
- 장점: 공간을 많이 차지하지 않음, 귀여움  
- 고려사항: 짧은 수명, 주기적 청소 필요

4. 토끼  
-

--

# 매개변수: temperature

**설명**: 출력의 무작위성을 제어합니다. 낮은 값은 출력을 더 결정론적으로 만들고, 높은 값은 무작위성을 증가시킵니다. \
**값 범위**: 0에서 1 \
**기본값**: 1 \
**예제**: temperature=0.7

높은 값은 모델이 더 창의적인 출력을 생성하도록 합니다. \
창의적인 응용 프로그램에는 0.9를, 명확한 답변이 필요한 경우에는 0(최대 확률 샘플링)을 시도하세요.

---
**참고**: 일반적으로 이 매개변수 또는 top_p를 조정하되 둘 다 동시에 조정하지 않는 것을 권장합니다.


In [4]:
def call_openai(num_times, prompt, temperature=0.75, use_seed=False):
    for i in range(num_times):
        if use_seed:
            response = client.chat.completions.create(
                model=CHAT_COMPLETIONS_MODEL,
                messages = [{"role":"system", "content":"You are a helpful assistant."},
                            {"role":"user","content": prompt}],
                    max_tokens=60,
                    seed=SEED,
                    temperature = temperature
            )
        else:
            response = client.chat.completions.create(
                model=CHAT_COMPLETIONS_MODEL,
                messages = [{"role":"system", "content":"You are a helpful assistant."},
                            {"role":"user","content": prompt}],
                    max_tokens=60,
                    temperature = temperature
            )
        print(response.choices[0].message.content)

In [5]:
# Without seed and temperature, the response is different each time
call_openai(10, '최고의 반려동물은 ')

“최고의 반려동물은 무엇인가요?”라는 질문에는 정답이 없습니다. 사람마다 환경, 생활 방식, 취향, 알레르기, 가족 구성원 등에 따라 최고의 반려동물은 다를 수 있습니다. 아래는 몇 가지 대표적인 반려
"최고의 반려동물"은 사람마다 다르게 느낄 수 있습니다. 각 반려동물의 특성, 라이프스타일, 환경, 그리고 개인의 취향에 따라 최고의 반려동물은 달라집니다. 대표적인 반려동물
최고의 반려동물은 사람마다 다를 수 있습니다. 라이프스타일, 집의 크기, 시간, 알레르기, 선호도 등에 따라 최적의 선택이 달라지기 때문입니다. 아래에 몇 가지 대표적인 반려동물과
"최고의 반려동물"은 사람마다 다르게 생각할 수 있습니다. 라이프스타일, 환경, 취향에 따라 다르기 때문입니다. 대표적인 반려동물과 그 특징을 알려드릴게요.

1. 강아지  
- 장
최고의 반려동물은 사람마다 다를 수 있습니다. 라이프스타일, 취향, 환경에 따라 다른 동물들이 최고의 반려동물이 될 수 있기 때문입니다. 몇 가지 예를 들어 설명해드릴게요:

1. 강아지  
활
"최고의 반려동물"은 사람마다 다르게 느낄 수 있습니다. 각 반려동물은 특징과 장점, 단점이 다르기 때문입니다. 아래에 대표적인 반려동물의 특징을 정리해드릴게요.

1
최고의 반려동물은 사람마다 다르게 생각할 수 있습니다. 각각의 반려동물은 고유한 특성과 장단점이 있기 때문입니다. 아래에 몇 가지 주요 반려동물과 그 특징을 소개해드릴게요.

1. 개
"최고의 반려동물"이라는 질문에 대한 답은 사람마다 다를 수 있습니다. 각자의 생활환경, 성격, 취향 등이 다르기 때문입니다. 몇 가지 예를 들어볼게요:

1. **강아지**  
사람과 교
"최고의 반려동물"은 사람마다 다르게 생각할 수 있습니다. 각자의 생활 방식, 취향, 환경에 따라 최고의 반려동물은 달라지기 때문입니다. 몇 가지 예를 들어보면:

1. **강아지**  
-
최고의 반려동물은 사람마다 다를 수 있습니다. 각자의 생활 방식, 환경, 취향에 따라 최고의 반려동물은 달라집니다. 몇 

In [ ]:
# Now using a seed and 0 temperature, the response is the much more consisitent
call_openai(10, '최고의 반려동물은 ', temperature = 0, use_seed=True)

최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개해드리겠습니다.

1. **강아지**: 강아지는 충성스럽고 사회적이며
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개하겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어 크기
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개하겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어 크기
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개해드리겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개해드리겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개하겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어 크기
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개하겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어 크기
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개하겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어 크기
최고의 반려동물은 개인의 생활 방식, 취향, 필요에 따라 다를 수 있습니다. 몇 가지 인기 있는 반려동물과 그 특징을 소개하겠습니다:

1. **개**: 충성스럽고 사회적이며 다양한 품종이 있어 크기

# 매개변수: n
**설명**: 각 프롬프트에 대해 생성할 응답의 수를 지정합니다. \
**기본값**: 1 \
**예제**: n = 3 

---
**참고**: 이 매개변수는 여러 응답을 생성하므로 토큰 할당량을 빠르게 소모할 수 있습니다. 신중하게 사용하고 max_tokens 및 stop 설정이 적절한지 확인하세요.

In [6]:
response = client.chat.completions.create(
            model=CHAT_COMPLETIONS_MODEL,
            messages = [{"role":"system", "content":"You are a helpful assistant."},
                        {"role":"user","content": "최고의 반려동물은"}],
                max_tokens=60,
                n=2
        )

for index, c in enumerate(response.choices):
    print(index, c.message.content)
    print("\n" + "-"*80 + "\n")

0 "최고의 반려동물"은 사람마다 다르게 생각할 수 있습니다. 각자의 라이프스타일, 성격, 환경, 취향에 따라 가장 잘 맞는 반려동물이 달라지기 때문입니다. 아래에 대표적인 반려동물과 각각

--------------------------------------------------------------------------------

1 "최고의 반려동물"은 사람마다 다르게 느낄 수 있는 질문입니다. 각 반려동물마다 특징과 장점, 그리고 단점이 있기 때문입니다. 아래에 몇 가지 인기 있는 반려동물의 특성과 추천 이유를 소개해드

--------------------------------------------------------------------------------



# 매개변수: presence_penalty
**설명**: 텍스트에 이미 나타난 토큰을 기반으로 새 토큰에 페널티를 부여하여 모델이 새로운 토큰을 사용하도록 유도합니다. \
**값 범위**: -2.0에서 2.0 \
**기본값**: 0 \
**예제**: presence_penalty=0.5

In [7]:
def call_openai_with_presence_penalty(presence_penalty):
    response = client.chat.completions.create(
          model=CHAT_COMPLETIONS_MODEL,
          messages = [{"role":"system", "content":"You are a helpful assistant."},
                    {"role":"user","content": "최고의 반려동물은 "}],
                    max_tokens=60,
                    presence_penalty=presence_penalty, 
                    
    )
    return response.choices[0].message.content

# Generate with different presence_penalty values
penalties = [0, 0.5, 1.0, 1.5, 2.0]
for penalty in penalties:
    print(f"Presence Penalty: {penalty}\n")
    print(call_openai_with_presence_penalty(penalty))
    print("\n" + "-"*80 + "\n")

Presence Penalty: 0



“최고의 반려동물”은 사람마다 다를 수 있습니다. 각자의 생활 방식, 성격, 취향, 환경 등에 따라 가장 잘 맞는 반려동물이 다릅니다. 여기 몇 가지 대표적인 반려동물을 소개해드릴게요:

1

--------------------------------------------------------------------------------

Presence Penalty: 0.5

"최고의 반려동물"은 사람마다 다르게 느낍니다. 개인의 라이프스타일, 취향, 거주 환경, 알레르기 등 여러 요소에 따라 달라질 수 있습니다. 예를 들어:

- 바쁜 직장인: 관리

--------------------------------------------------------------------------------

Presence Penalty: 1.0

"최고의 반려동물"은 사람마다 다를 수 있습니다. 라이프스타일, 주거 환경, 성격, 책임감 등 여러 요소에 따라 최적의 반려동물이 달라지죠. 몇 가지 예시를 들어볼게요:

1.

--------------------------------------------------------------------------------

Presence Penalty: 1.5

최고의 반려동물은 사람마다 다르게 생각할 수 있습니다. 왜냐하면 각자의 라이프스타일, 환경, 취향에 따라 어울리는 반려동물이 다르기 때문입니다. 아래에 몇 가지 예시와 특징을 소개해드릴게

--------------------------------------------------------------------------------

Presence Penalty: 2.0

“최고의 반려동물”은 사람마다 다르게 생각할 수 있습니다. 각각의 라이프스타일, 환경, 선호도 등이 영향을 미치기 때문이에요. 몇 가지 주요 반려동물을 소개하면서 장단점을 알려드릴게요.

1.

--------------------------------------------------------

# 매개변수: frequency_penalty
**설명**: 텍스트에 이미 나타난 빈도를 기반으로 새 토큰에 페널티를 부여하여 동일한 줄을 반복할 가능성을 줄입니다. \
**값 범위**: -2.0에서 2.0 \
**기본값**: 0 \
**예제**: frequency_penalty=0.5

#### 탐색할 사용 사례
1. **응답 비교** \
여러 응답을 생성하여 사용 사례에 가장 적합한 응답을 선택하세요.

2. **다양성 증가** \
다양한 응답을 얻기 위해 여러 응답을 생성하세요. 이는 창의적인 응용 프로그램에 유용합니다.

3. **강건성 향상** \
여러 응답을 생성하여 일관성과 정확성을 보장하세요.

#### 모범 사례
1. **프롬프트 길이 최적화** \
프롬프트를 간결하지만 정보가 충분하도록 유지하여 모델이 충분한 컨텍스트를 가지도록 하세요.

2. **Temperature 및 Top_p 조정** \
결정론적 응답과 창의적 응답 간의 균형을 맞추기 위해 이 매개변수를 사용하세요.

3. **토큰 사용량 모니터링** \
max_tokens 매개변수를 신중히 설정하여 비용과 응답 길이를 관리하세요.

4. **중지 시퀀스 사용** \
모델이 텍스트 생성을 중지해야 할 위치를 정의하여 원하는 컨텍스트 내에서 출력을 제어하세요.

5. **여러 응답 생성** \
n 매개변수를 사용하여 여러 응답을 생성하고 필요에 가장 적합한 응답을 선택하세요.